# E-Scooter Trip Analysis**Dataset**: anonymised scooter trip data, Berlin, 21 Jun 2019 – 07 Oct 2019 (≈ 232k rides, 82k riders, 2.9k scooters).**Audience framing**: a budget & forecasting analyst at a shared-mobility operator. The goal is not only to answer the five case questions, but to build the KPI set a Courier Analytics team would monitor, with causal-inference-friendly framings and clear "so-whats".## Contents1. Load & sanity-check2. Ride statistics (Q1)3. Distribution over time (Q2)4. Rolling averages (Q3)5. Retention & cohorts (Q4)6. Geospatial demand & imbalance (Q5)7. Fleet utilisation, pricing, and forecasting — beyond the brief8. So-whats: prioritised recommendations

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathsns.set_theme(style='whitegrid', context='talk')pd.set_option('display.float_format', lambda v: f'{v:,.3f}')DATA_PATH = Path('../data/trips_clean.csv')   # exported from Snowflakedf = pd.read_csv(DATA_PATH, parse_dates=['START_TIME_LOCAL','END_TIME_LOCAL','PREVIOUS_END_TIME'])print(df.shape)df.head(2)

## 1 · Sanity checks & cleaningQuick integrity look: null counts, outliers, cancel-like rides. Drop rides that look like unlock-then-cancel (distance < 0.05 km AND duration < 2 min) because they inflate averages and hide the real usage pattern.

In [ ]:
print('date range :', df['START_TIME_LOCAL'].min(), '→', df['START_TIME_LOCAL'].max())print('unique riders    :', df['USER_HASH'].nunique())print('unique scooters  :', df['SCOOTER_HASH'].nunique())print()mask_trivial = (df['TRIP_DISTANCE'] < 0.05) & (df['RIDE_DURATION'] < 2)print(f'trivial rides: {mask_trivial.sum():,}  ({mask_trivial.mean()*100:.1f}%)')df = df[~mask_trivial & df['RIDE_DURATION'].between(0.5, 120)].copy()df['START_DATE'] = df['START_TIME_LOCAL'].dt.datedf.shape

## 2 · Ride statistics (Q1)Headline KPIs the team would present weekly.

In [ ]:
kpi = pd.Series({    'total_rides':             len(df),    'unique_riders':           df['USER_HASH'].nunique(),    'unique_scooters':         df['SCOOTER_HASH'].nunique(),    'total_revenue_eur':       df['PRICE'].sum(),    'avg_price_per_ride':      df['PRICE'].mean(),    'median_price_per_ride':   df['PRICE'].median(),    'avg_duration_min':        df['RIDE_DURATION'].mean(),    'median_duration_min':     df['RIDE_DURATION'].median(),    'avg_distance_km':         df['TRIP_DISTANCE'].mean(),    'median_distance_km':      df['TRIP_DISTANCE'].median(),    'rides_per_rider':         len(df)/df['USER_HASH'].nunique(),    'rides_per_scooter':       len(df)/df['SCOOTER_HASH'].nunique(),})kpi.round(3)

In [ ]:
# Weekday vs weekend splitsplit = df.groupby('DAY_TYPE2').agg(    rides=('RIDE_HASH','count'),    riders=('USER_HASH','nunique'),    avg_price=('PRICE','mean'),    avg_duration=('RIDE_DURATION','mean'),    avg_distance=('TRIP_DISTANCE','mean'),    revenue=('PRICE','sum'),).round(2)split

**Observation.** Weekends have ~39% fewer trips than weekdays but trips are ~15% longer in duration and ~8% longer in distance. That's classic leisure-vs-commute signature.

## 3 · Distribution over time (Q2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,4.5))by_hour = df.groupby('START_HOUR').size()axes[0].bar(by_hour.index, by_hour.values, color='#1F4E79')axes[0].set_title('Rides by hour of day'); axes[0].set_xlabel('Hour'); axes[0].set_xticks(range(0,24))dow_order = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']by_dow = df['DAY_ABBR'].value_counts().reindex(dow_order)colors = ['#1F4E79']*5 + ['#F06E00']*2axes[1].bar(by_dow.index, by_dow.values, color=colors)axes[1].set_title('Rides by day of week')plt.tight_layout(); plt.show()

In [ ]:
heat = df.pivot_table(index='START_HOUR', columns='DAY_ABBR', values='RIDE_HASH', aggfunc='count').reindex(columns=dow_order)plt.figure(figsize=(9,7))sns.heatmap(heat, cmap='Blues', cbar_kws={'label':'Rides'}, linewidths=.3, linecolor='white')plt.title('Demand heatmap — hour × day of week'); plt.ylabel('Hour'); plt.xlabel('')plt.show()

**Observation.** Twin weekday peaks at 08:00 and 17:00-19:00 (commute), weekend peak around midnight–02:00 (nightlife). Demand pattern suggests separate supply playbooks for weekday AM, weekday PM, and weekend-night shifts.

## 4 · Rolling averages (Q3)

In [ ]:
daily = df.groupby('START_DATE').agg(    rides=('RIDE_HASH','count'),    riders=('USER_HASH','nunique'),    revenue=('PRICE','sum'),).reset_index()daily['START_DATE'] = pd.to_datetime(daily['START_DATE'])daily = daily.set_index('START_DATE').sort_index()daily['rides_7d']   = daily['rides'].rolling(7).mean()daily['rides_28d']  = daily['rides'].rolling(28).mean()daily['revenue_7d'] = daily['revenue'].rolling(7).mean()fig, ax = plt.subplots(1,1, figsize=(13,4.5))ax.plot(daily.index, daily['rides'], color='#8FAADC', alpha=.6, label='Daily rides')ax.plot(daily.index, daily['rides_7d'], color='#1F4E79', lw=2.4, label='7-day rolling avg')ax.plot(daily.index, daily['rides_28d'], color='#F06E00', lw=2.4, label='28-day rolling avg')ax.set_title('Daily rides with rolling averages'); ax.set_ylabel('Rides'); ax.legend(frameon=False)plt.show()

**Observation.** Growth through July–August, plateau in September, clear September-end drop-off as autumn weather begins. Temperature / precipitation is a likely driver and should be folded into any forecast.

## 5 · Retention & cohorts (Q4)

In [ ]:
rides_per_user = df.groupby('USER_HASH').size()tot = rides_per_user.shape[0]rows = []for n in [1,2,3,4,5,6,7,8,9,10,15,20]:    r = int((rides_per_user >= n).sum())    rows.append({'ride_number': n, 'riders_reaching_n': r, 'pct_of_riders': round(r/tot*100,2)})pd.DataFrame(rows)

In [ ]:
# Step-down conversion P[n+1 | n]step = pd.DataFrame(rows)step['step_conversion_pct'] = (step['riders_reaching_n'].shift(-1) / step['riders_reaching_n'] * 100).round(1)step

In [ ]:
# Weekly cohort retention matrixfirst = df.groupby('USER_HASH')['START_TIME_LOCAL'].min().rename('first_ts').reset_index()first['cohort_month'] = first['first_ts'].dt.to_period('M').astype(str)df2 = df.merge(first[['USER_HASH','first_ts','cohort_month']], on='USER_HASH')df2['weeks_since_first'] = ((df2['START_TIME_LOCAL'] - df2['first_ts']).dt.days // 7)coh = df2.pivot_table(index='cohort_month', columns='weeks_since_first',                     values='USER_HASH', aggfunc='nunique').fillna(0)cohort_size = coh.iloc[:,0]coh_pct = coh.div(cohort_size, axis=0) * 100coh_pct.iloc[:,:12].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))for cm in ['2019-06','2019-07','2019-08','2019-09']:    if cm in coh_pct.index:        v = coh_pct.loc[cm].iloc[:12]        ax.plot(v.index, v.values, marker='o', label=f'Cohort {cm}')ax.set_title('Weekly retention curves by acquisition-month cohort')ax.set_xlabel('Weeks since first ride'); ax.set_ylabel('% of cohort active'); ax.legend(frameon=False)plt.show()

**Observation.** Only ~54% of riders take a second ride; roughly half of those take a third. By week 4 fewer than 5% of each cohort is still active. A *Senior DS / Courier* framing: this is the exact funnel a retention-focused experimentation programme targets. Candidate interventions:- Re-engagement push after the first 24h if no second ride is initiated (A/B test with matched controls).- Pricing: bundled minutes or subscription unlock-fee waiver for users who took 1 ride but zero return.- Surface a personalised "saved route" when a rider returns to the app.

## 6 · Geospatial demand & imbalance (Q5)

In [ ]:
# Bin to ~350m × 550m cellsdf['lat_bin'] = (df['START_LAT']/0.005).round()*0.005df['lon_bin'] = (df['START_LONG']/0.005).round()*0.005grid = df.groupby(['lat_bin','lon_bin']).size().reset_index(name='rides')grid.sort_values('rides', ascending=False).head(10)

In [ ]:
plt.figure(figsize=(10,8))plt.scatter(grid['lon_bin'], grid['lat_bin'],            c=grid['rides'], cmap='magma',            s=np.clip(grid['rides']/8, 4, 300), alpha=.85, edgecolor='none')plt.colorbar(label='Rides started')plt.title('Demand map — start-location density'); plt.xlabel('Longitude'); plt.ylabel('Latitude')plt.show()

In [ ]:
# Net flow: ends minus starts per cell → supply imbalancedf['end_lat_bin'] = (df['END_LAT']/0.005).round()*0.005df['end_lon_bin'] = (df['END_LONG']/0.005).round()*0.005starts = df.groupby(['lat_bin','lon_bin']).size().reset_index(name='starts')ends = df.groupby(['end_lat_bin','end_lon_bin']).size().reset_index(name='ends').rename(    columns={'end_lat_bin':'lat_bin','end_lon_bin':'lon_bin'})bal = starts.merge(ends, on=['lat_bin','lon_bin'], how='outer').fillna(0)bal['net_flow'] = bal['ends'] - bal['starts']bal2 = bal[(bal['starts']+bal['ends']) > 50]vmax = max(abs(bal2['net_flow'].quantile(.99)), abs(bal2['net_flow'].quantile(.01)))plt.figure(figsize=(10,8))plt.scatter(bal2['lon_bin'], bal2['lat_bin'], c=bal2['net_flow'], cmap='RdBu',            s=np.clip((bal2['starts']+bal2['ends'])/15,4,260),            alpha=.85, edgecolor='none', vmin=-vmax, vmax=vmax)plt.colorbar(label='Net flow (ends − starts)')plt.title('Supply/demand imbalance by cell\nblue = supply accumulates, red = supply depletes')plt.show()

**Observation.** Several cells show consistent depletion (red, ends < starts). Those are deploy-back candidates. Conversely, the blue accumulation cells are pickup candidates for overnight rebalancing. Pair this with time-of-day slicing in production to drive the rebalancing schedule.

## 7 · Beyond the brief: fleet utilisation, pricing, forecasting

In [ ]:
# Fleet utilisation — active hours per scooter per weekdf['week_start'] = df['START_TIME_LOCAL'].dt.to_period('W').apply(lambda r: r.start_time)scoot_week = df.groupby(['SCOOTER_HASH','week_start'])['RIDE_DURATION'].sum().reset_index()scoot_week['active_hours'] = scoot_week['RIDE_DURATION']/60scoot_week['util_pct'] = scoot_week['active_hours']/(24*7)*100fleet = scoot_week.groupby('week_start')['util_pct'].agg(['mean','median',lambda x: x.quantile(.9)]).rename(columns={'<lambda_0>':'p90'}).reset_index()fleet

In [ ]:
# Pricing curve: price vs ride-duration bucket (proxy for elasticity questions)pd_ = df[df['RIDE_DURATION']<60].copy()pd_['dur_bucket'] = pd.cut(pd_['RIDE_DURATION'], bins=range(0,61,5))curve = pd_.groupby('dur_bucket', observed=True)['PRICE'].agg(['mean','count','median']).reset_index()curve['dur_bucket'] = curve['dur_bucket'].astype(str)curve.head()

In [ ]:
# Simple Holt-Winters revenue forecast (weekly seasonality)from statsmodels.tsa.holtwinters import ExponentialSmoothingrev = daily['revenue'].copy()rev.index = pd.to_datetime(rev.index)hw = ExponentialSmoothing(rev, seasonal_periods=7, trend='add', seasonal='add').fit()fc = hw.forecast(28)fig, ax = plt.subplots(figsize=(13,5))ax.plot(rev.index, rev.values, color='#1F4E79', lw=1.8, label='Actual')ax.plot(fc.index, fc.values, color='#F06E00', lw=2.2, ls='--', label='HW forecast (+28d)')ax.set_title('Revenue — Holt-Winters forecast'); ax.legend(frameon=False); plt.show()

## 8 · So-whats**Budget & forecasting view (the brief)**1. **Peak-driven capacity planning.** Target fleet density around the two weekday commuter peaks (08:00 and 17:00–19:00) and the weekend night peak (midnight–02:00). Shift overnight rebalancing to get scooters into depletion cells before 07:30.2. **Summer concentration.** ~70% of rides happen in July–August. The autumn tail drops fast: build a month-specific marketing + retention budget curve rather than a flat monthly spend.3. **Retention is the single biggest lever.** Only ~54% of users return for a 2nd ride. A 5 percentage-point lift at that step compounds through the funnel and materially changes LTV. This is a clean A/B-testable problem with a large n.**Operations / product framings a Senior DS would raise next**4. **Rebalancing ROI.** The net-flow map identifies cells whose depletion is chronic. Pair with idle-time and time-of-day to score rebalancing trips by expected marginal rides recovered.5. **Pricing experimentation.** Price curve vs duration is roughly linear; test a flat unlock-fee waiver for dormant riders (quasi-experiment with matched controls). Track treatment LTV vs control over 4 weeks.6. **Cancel-like rides (1.2%).** Small share but they hurt unit economics and rider experience. Diagnose by scooter_hash and geo cluster to spot defective units.**Data & instrumentation gaps**- No weather join — add OpenWeather API to de-seasonalise revenue forecasts.- No acquisition channel — block all CAC/retention causal questions. Ask product to stamp source on first ride.